# Protein-ligand interactions with ProLIF

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yboulaamane/cadd-labs/blob/main/protein_ligand_interactions.ipynb)

Docking or an MD run tells you *where* a molecule sits. This notebook tells you *why* it stays:
which residues it actually touches, and how. It reads the crystal structure of human MAO-B with
safinamide ([2V5Z](https://www.rcsb.org/structure/2V5Z)) and uses
[ProLIF](https://prolif.readthedocs.io/) to find the hydrogen bonds, hydrophobic contacts and
pi-stacking between ligand and protein.

**What you'll do**

- Read the structure and prepare it for analysis (bond orders fixed from templates).
- Compute an **interaction fingerprint**: which residue makes which kind of contact.
- See it three ways, as a table, a 2D interaction diagram and an interactive 3D view.

The same code works on a pose you docked yourself (see the
[docking notebook](docking_autodock_vina.ipynb)): just read that pose as the ligand.

Runtime: CPU is fine.

## Setup

ProLIF brings in RDKit and MDAnalysis. py3Dmol draws the 3D view.

In [ ]:
%pip install -q prolif py3Dmol

In [ ]:
import warnings

import prolif as plf
from prolif.io import MoleculeStandardizer
from rdkit import Chem, RDLogger

RDLogger.DisableLog("rdApp.*")

In [ ]:
LIGAND_SMILES = {'SAG': 'C[C@H](NCc1ccc(OCc2cccc(F)c2)cc1)C(N)=O', 'FAD': 'Cc1cc2nc3c(=O)[nH]c(=O)nc-3n(CC(O)C(O)C(O)COP(=O)([O-])OP(=O)([O-])OCC3OC(n4cnc5c(N)ncnc54)C(O)C3O)c2cc1C'}

In [ ]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem


def read_pdb_lines(pdb_path):
    """Return (protein_lines, hetero_by_residue). Keeps the first altloc, drops waters."""
    protein, hetero = [], {}
    for line in open(pdb_path):
        record = line[:6].strip()
        if record not in ("ATOM", "HETATM"):
            continue
        if line[16] not in " A":                 # keep only the first alternate location
            continue
        resname = line[17:20].strip()
        if record == "ATOM":
            protein.append(line)
        elif resname != "HOH":                    # a small molecule, not a crystal water
            key = (resname, line[21], line[22:26].strip())
            hetero.setdefault(key, []).append(line)
    return protein, hetero


def residue_from_lines(lines, smiles):
    """Build an RDKit molecule from PDB atom lines, taking bond orders from `smiles`.

    PDB files store no bond orders. RDKit guesses bonds from distances; this function
    then sets their orders, charges and hydrogens from the SMILES template. Guessed bonds
    the template does not have (such as the spurious P-P bond RDKit adds across the two
    phosphates of FAD) are removed before matching.
    """
    block = "".join(l for l in lines if l[76:78].strip() != "H")
    guessed = Chem.RWMol(Chem.MolFromPDBBlock(block, removeHs=True, sanitize=False))
    guessed.UpdatePropertyCache(strict=False)
    Chem.FastFindRings(guessed)

    template = Chem.MolFromSmiles(smiles)
    skeleton = Chem.RWMol(template)
    for bond in skeleton.GetBonds():
        bond.SetBondType(Chem.BondType.SINGLE)
        bond.SetIsAromatic(False)
    for atom in skeleton.GetAtoms():
        atom.SetFormalCharge(0)
        atom.SetIsAromatic(False)
    skeleton.UpdatePropertyCache(strict=False)

    match = guessed.GetSubstructMatch(skeleton)
    if not match or len(match) != guessed.GetNumAtoms():
        raise ValueError("The atoms in the PDB residue do not match the SMILES given for it")
    keep = {frozenset((match[b.GetBeginAtomIdx()], match[b.GetEndAtomIdx()]))
            for b in template.GetBonds()}
    for bond in list(guessed.GetBonds()):
        if frozenset((bond.GetBeginAtomIdx(), bond.GetEndAtomIdx())) not in keep:
            guessed.RemoveBond(bond.GetBeginAtomIdx(), bond.GetEndAtomIdx())

    mol = AllChem.AssignBondOrdersFromTemplate(template, guessed.GetMol())
    Chem.AssignStereochemistryFrom3D(mol)
    return Chem.AddHs(mol, addCoords=True)

## Download 2V5Z and split it

The protein of chain A is written on its own; the FAD cofactor and the safinamide ligand
are pulled out with their bond orders assigned from the templates.

In [ ]:
!wget -q https://files.rcsb.org/download/2V5Z.pdb -O 2V5Z.pdb
protein_lines, hetero = read_pdb_lines("2V5Z.pdb")
chain_a = [line for line in protein_lines if line[21] == "A"]
open("protein_A.pdb", "w").writelines(chain_a + ["END\n"])

fad_lines = next(v for (name, chain, _), v in hetero.items() if name == "FAD" and chain == "A")
sag_lines = next(v for (name, chain, _), v in hetero.items() if name == "SAG" and chain == "A")
fad = Chem.RemoveHs(residue_from_lines(fad_lines, LIGAND_SMILES["FAD"]))
ligand_mol = Chem.RemoveHs(residue_from_lines(sag_lines, LIGAND_SMILES["SAG"]))
print("FAD atoms:", fad.GetNumAtoms(), "  ligand atoms:", ligand_mol.GetNumAtoms())

## Build the receptor ProLIF can read

ProLIF needs correct bond orders for every residue. `MoleculeStandardizer` fixes the
protein's from its built-in amino-acid templates; the FAD cofactor, already fixed above, is
added to it. The crystal structure has no hydrogens, so the fingerprint below runs in
**implicit-hydrogen mode**, which infers hydrogen-bond geometry without a separate
protonation step. (It prints warnings about atom counts for charged and terminal residues;
they are informational.)

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    protein = MoleculeStandardizer()("protein_A.pdb")

receptor = plf.Molecule.from_rdkit(Chem.CombineMols(protein, fad))
ligand = plf.Molecule.from_rdkit(ligand_mol)
print(f"Receptor: {receptor.n_residues} residues (protein + FAD)")

## Compute the interaction fingerprint

ProLIF checks every residue near the ligand for each interaction type and records which
ones are present.

In [ ]:
fp = plf.Fingerprint(implicit_hydrogens=True)
fp.run_from_iterable([ligand], receptor, progress=False)

table = fp.to_dataframe(index_col="Ligand")
table.columns = table.columns.droplevel("ligand")
interactions = table.T
interactions.columns = ["present"]
interactions

The residues that safinamide contacts. In MAO-B the two tyrosines of the "aromatic cage"
(Tyr398 and Tyr435) sandwich the ligand in front of the FAD, and Cys172 and the gating
residue Ile199 line the cavity. Whether your run reproduces the published contacts is a good
check of a docked pose.

In [ ]:
residues = sorted({residue for residue, _ in table.columns})
print(f"{len(residues)} residues contacted:")
print(", ".join(residues))

## 2D interaction diagram

`plot_lignetwork` draws the ligand with a node per contacting residue, coloured by
interaction type. It is interactive; drag the nodes to tidy the layout.

In [ ]:
fp.plot_lignetwork(ligand, kind="frame", frame=0)

## 3D view

The same interactions shown on the structure: the ligand, the residues it touches and dashed
lines for the interactions.

In [ ]:
fp.plot_3d(ligand, receptor, frame=0, size=(700, 500))

## Compare several ligands or poses

`run_from_iterable` takes a list, so you can fingerprint many poses or compounds at once and
compare their contacts row by row. For a docked set, read each pose with
`plf.Molecule.from_rdkit` and pass them together; the resulting table has one row per pose,
which makes it easy to see which contacts a modification keeps or loses.

## Takeaways

- An interaction fingerprint turns "it binds" into a concrete list of contacts you can check,
  compare and defend. It is how you tell a plausible pose from a lucky-looking one.
- For a docked pose, the test is simple: does it reproduce the key contacts the crystal ligand
  makes? Here, the aromatic-cage tyrosines and the contacts near the FAD are the ones to look for.
- Because it is just a table under the hood, you can fingerprint a whole series and spot exactly
  which interaction a chemical change gains or loses.

**Try it yourself:** point the ligand at one of your own docked poses and compare its row with the crystal ligand's.